In [21]:
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [39]:
llm.invoke("How will the weather be in munich today?")

AIMessage(content="I'm unable to provide real-time weather updates. For the most current weather information in Munich, I recommend checking a reliable weather website or a weather app.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 17, 'total_tokens': 47, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2d82c05f26', 'id': 'chatcmpl-DFasAoOhJzxyhuPJS1pptJo6O4Fyy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cb7bb-48dc-75a2-b26e-cd1a759141b4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 30, 'total_tokens': 47, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_detail

In [40]:
from langchain_core.tools import tool


@tool
def fake_weather_api(city: str) -> str:
    """
    Check the weather in a specified city.

    Args:
        city (str): The name of the city where you want to check the weather.

    Returns:
        str: A description of the current weather in the specified city.
    """
    return " sunny,22°C,"


tools = [fake_weather_api]

In [32]:
llm_with_tools = llm.bind_tools(tools)


In [44]:
result = llm_with_tools.invoke("How will the weather be in munich today?")
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 90, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_414ba99a04', 'id': 'chatcmpl-DFawNzR63Wyz5CCU2zNwUYkrvH1jb', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb7bf-42e3-74b3-b2dc-96fb2f60b639-0', tool_calls=[{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_sACd3YkZYNF72nw4LJmbnhLA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 16, 'total_tokens': 106, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [46]:

result.pretty_print()
result.tool_calls

================================== Ai Message ==================================
Tool Calls:
  fake_weather_api (call_sACd3YkZYNF72nw4LJmbnhLA)
 Call ID: call_sACd3YkZYNF72nw4LJmbnhLA
  Args:
    city: Munich


[{'name': 'fake_weather_api',
  'args': {'city': 'Munich'},
  'id': 'call_sACd3YkZYNF72nw4LJmbnhLA',
  'type': 'tool_call'}]

In [38]:
result.tool_calls

[{'name': 'fake_weather_api',
  'args': {'city': 'Munich'},
  'id': 'call_DBL1NJZ8XbghKCTGdES7mer9',
  'type': 'tool_call'}]

In [54]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        "How will the weather be in munich today?"
    )
]
llm_output = llm_with_tools.invoke(messages)
messages.append(llm_output)

In [48]:
messages = [
    HumanMessage(
        "How will the weather be in munich today?"
    )
]

In [49]:
llm_output = llm_with_tools.invoke(messages)

In [59]:
llm_output.tool_calls


[{'name': 'fake_weather_api',
  'args': {'city': 'Munich'},
  'id': 'call_flxxjrp825G5nJBs6ZpX6L9R',
  'type': 'tool_call'}]

In [55]:
messages

[HumanMessage(content='How will the weather be in munich today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 90, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_414ba99a04', 'id': 'chatcmpl-DFb4aMLF5CkmHQ5H8oYP5ASVHzCYv', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb7c7-07e1-7902-8fa3-e254a8bb5bac-0', tool_calls=[{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_flxxjrp825G5nJBs6ZpX6L9R', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 16, 'total_tokens': 106, 'inp

In [56]:
tool_mapping = {"fake_weather_api": fake_weather_api}

In [60]:
for tool_call in llm_output.tool_calls:
    tool = tool_mapping[tool_call["name"].lower()]
    tool_output = tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))

In [19]:
messages

[HumanMessage(content='How will the weather be in munich today?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 90, 'total_tokens': 106, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_414ba99a04', 'id': 'chatcmpl-DFaT9hv6YRvUilrBDTAlim6yQc34f', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cb7a3-9a20-7b11-8338-22a74663f14e-0', tool_calls=[{'name': 'fake_weather_api', 'args': {'city': 'Munich'}, 'id': 'call_OfIAac3vEVdFMoUtYoilOeqY', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 90, 'output_tokens': 16, 'total_tokens': 106, 'inp

In [61]:
llm_with_tools.invoke(messages)

AIMessage(content='The weather in Munich today is sunny with a temperature of 22°C.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 120, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_414ba99a04', 'id': 'chatcmpl-DFb95wD7S4n76dKYRABrdu5zxD5YE', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019cb7cb-496f-7dd0-9512-d8ea4bd0c57c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 120, 'output_tokens': 16, 'total_tokens': 136, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})